# 00 — Environment setup
Lightweight Glaucoma Detection via Ensemble Learning

Run this notebook at the beginning of each **new runtime**. It mounts Drive on Colab, optionally clones code, installs dependencies when explicitly enabled, checks TensorFlow/GPU execution, and writes resolved session settings. It does not download datasets, construct pretrained models or start training.

Use Python 3.10–3.12. Choose a GPU accelerator in Colab's runtime settings or Kaggle's session options if available; CPU remains valid. A platform upgrade may require dependency compatibility work. See the repository README before running cells. Changing the accelerator can restart the runtime.

## 1. Platform, repository and storage settings
Edit the values below. Leave dataset and artifacts on separate paths. On Colab, persistent outputs belong under mounted Google Drive. Kaggle input data is read-only; outputs go in `/kaggle/working` and must be saved as a notebook version/exported. Kaggle does not use `google.colab.drive.mount`. No datasets are fetched.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

PLATFORM = "kaggle" if Path("/kaggle/working").is_dir() else ("colab" if Path("/content").is_dir() else "local")
REPO_URL = "https://github.com/YOUR_USERNAME/glaucoma_detection.git"  # replace
REPO_REF = "main"  # preferably a reproducible tag or commit
REPO_DIR = Path("/content/glaucoma_detection" if PLATFORM == "colab" else "/kaggle/working/glaucoma_detection" if PLATFORM == "kaggle" else str(Path.cwd() if (Path.cwd() / "configs/project.yaml").is_file() else Path.cwd().parent)).resolve()
CLONE_REPOSITORY = False  # enable only if REPO_DIR does not already contain your repo
INSTALL_DEPENDENCIES = False  # enable after reviewing the dependency manifest
MOUNT_DRIVE = PLATFORM == "colab"
RUN_NAME = "development"  # change for each independent experiment
print({"platform": PLATFORM, "python": sys.version, "repository": str(REPO_DIR)})
if not (3, 10) <= sys.version_info[:2] < (3, 13):
    raise RuntimeError("This project's dependency range requires Python 3.10–3.12; use a compatible runtime.")


## 2. Mount Google Drive (Colab only)
Authorize the mount when prompted. Only run trusted notebooks: mounting gives notebook code access to Drive. A local clone under `/content` is ephemeral; store datasets and derived artifacts on Drive. Kaggle skips this step. There is no automated Kaggle-to-Drive authentication or synchronization.

In [ ]:
if PLATFORM == "colab" and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive mount skipped; choose writable persistent storage explicitly.")


## 3. Obtain the GitHub repository
Create/push your repository using the README's GitHub instructions first. Set `REPO_URL` and `REPO_REF` above and enable cloning, or provide an existing clone. For private repositories, authenticate outside notebook source and never embed tokens in URLs/cells/outputs. Kaggle requires Internet access for GitHub and pip; an attached repository archive is an offline alternative. Existing clones are not reset or pulled automatically.

In [ ]:
if CLONE_REPOSITORY:
    if "YOUR_USERNAME" in REPO_URL:
        raise ValueError("Set your real GitHub repository URL first.")
    if REPO_DIR.exists():
        raise FileExistsError("Destination exists; inspect it or select another location instead of overwriting it.")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)
if not (REPO_DIR / "configs/project.yaml").is_file():
    raise FileNotFoundError("Set REPO_DIR to your project clone or enable cloning after configuring its URL.")
os.chdir(REPO_DIR)
os.environ["PROJECT_REPO"] = str(REPO_DIR)
print("Working directory:", Path.cwd())


## 4. Install dependencies
Enable `INSTALL_DEPENDENCIES` above for the first setup. This installs the shared research stack and notebook tools, never datasets or pretrained weights. The project's supported TensorFlow range may replace hosted preinstalled packages. **Restart the kernel/session after installation**, then rerun 00 with installation disabled. Do not repeatedly reinstall in an active TensorFlow session. Use a separate environment for notebook 08's converter dependencies; do not silently repair hosted CUDA/system packages.

In [ ]:
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-notebooks.txt"], check=True)
    raise RuntimeError("Installation complete. Restart the kernel, disable INSTALL_DEPENDENCIES and rerun 00.")
else:
    print("Dependency installation skipped. Enable it if imports fail.")


## 5. TensorFlow and GPU verification
The reusable helper enables GPU memory growth when possible and executes a tiny matrix multiplication. It does not build a model. CPU fallback is supported; `nvidia-smi` is an optional independent diagnostic. If an accelerator was selected but TensorFlow sees no GPU, check the runtime and dependency compatibility before future training.

In [ ]:
import json
import shutil
from glaucoma.utils.notebook_runtime import configure_workspace, save_session, verify_tensorflow

tf_info = verify_tensorflow()
print(json.dumps(tf_info, indent=2))
if not tf_info["physical_gpus"]:
    print("No TensorFlow GPU detected; using CPU.")
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False, timeout=30)


## 6. Configure dataset and persistent outputs
Edit these paths to your actual existing labeled ACRIMA download. No dataset folder is created. Confirm the folder-to-label mapping or annotation path in `configs/project.yaml`; unknown annotations/patient IDs must remain unknown. `DATASET_ROOT` is respected if exported. Each run gets checkpoints (core/baselines separately), metrics, histories, heatmaps, exports, manifests and environment records.

Google Drive must be mounted **before** creating output folders. For Kaggle, attach a private dataset through the notebook UI and replace the input path. The default output is not Drive: save a notebook version with outputs, then download the output bundle and upload/copy it to Drive yourself. Do not assume an interactive Kaggle session persists forever.

In [ ]:
if PLATFORM == "colab":
    if not Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Mount Google Drive before configuring persistent outputs.")
    default_dataset = "/content/drive/MyDrive/glaucoma_project/data/raw/ACRIMA"
    default_artifacts = "/content/drive/MyDrive/glaucoma_project/artifacts"
elif PLATFORM == "kaggle":
    default_dataset = "/kaggle/input/YOUR_ACRIMA_DATASET/ACRIMA"  # EDIT to attached dataset path
    default_artifacts = "/kaggle/working/glaucoma_artifacts"
else:
    default_dataset = str(REPO_DIR / "data/raw/ACRIMA")
    default_artifacts = str(REPO_DIR / "artifacts")

DATASET_ROOT = Path(os.environ.get("DATASET_ROOT", default_dataset))
ARTIFACTS_ROOT = Path(os.environ.get("ARTIFACTS_ROOT", default_artifacts))
CONFIG_PATH = REPO_DIR / "configs/project.yaml"
cfg = configure_workspace(CONFIG_PATH, DATASET_ROOT, ARTIFACTS_ROOT, RUN_NAME)
SESSION_FILE = Path(cfg["artifacts"]["environment"]) / "session.json"
save_session(cfg, SESSION_FILE)
os.environ["GLAUCOMA_SESSION"] = str(SESSION_FILE)
print("SESSION_FILE =", str(SESSION_FILE))
print(json.dumps({"dataset": str(DATASET_ROOT), "checkpoints": cfg["checkpoints"], "artifacts": cfg["artifacts"]}, indent=2))
if not DATASET_ROOT.is_dir():
    print("Dataset is not present. Set its path and provide your own download before notebook 01/02.")


## 7. Record the runtime and hand off
Save dependency versions, the source commit when available, and the tiny TensorFlow check alongside resolved configuration. These files contain local paths: keep them in private artifact storage. The session file lets later notebooks load the same configuration without relying on Python variables in another notebook. New kernels must import the editable package again if its installation is gone. Rerun 00 after a fresh hosted runtime and update `SESSION_FILE` in subsequent notebooks.

Future training code must save directly using `cfg['checkpoints']` and `cfg['artifacts']`, including histories/metrics. Future conversion must use `cfg['export']['output_root']`; copying approved core exports into the frontend is a later explicit step. Existing model/training/evaluation modules remain placeholders.

In [ ]:
environment_dir = Path(cfg["artifacts"]["environment"])
(environment_dir / "tensorflow.json").write_text(json.dumps(tf_info, indent=2) + "\n")
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], check=True, capture_output=True, text=True)
(environment_dir / "requirements-freeze.txt").write_text(freeze.stdout)
commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True)
(environment_dir / "source-commit.txt").write_text(commit.stdout.strip() + "\n" if commit.returncode == 0 else "Not a Git checkout\n")
print("Setup complete. No dataset processing, model construction or training was started.")
